# Annadata — Produce Quality CNN

**Runtime → Change runtime type → T4 GPU**

## Critical (Colab due diligence)
- Colab already has TensorFlow (~2.19–2.20) + pandas. **Do NOT `pip install tensorflow` or upgrade pandas.**
- That is what caused your protobuf / tf-keras / pandas conflicts.
- If you already broke the runtime: **Runtime → Disconnect and delete runtime**, reconnect, then run this notebook top → bottom.
- Only Cell 1 installs `kaggle` (tiny CLI).

Accept the dataset license once:  
https://www.kaggle.com/datasets/muhammad0subhan/fruit-and-vegetable-disease-healthy-vs-rotten → **I Agree**


In [ ]:
# Cell 1 — ONLY kaggle (never tensorflow / pandas)
!pip install -q kaggle
print("kaggle CLI ready")


In [ ]:
# Cell 2 — Kaggle auth → ~/.kaggle/kaggle.json
import json, os
from pathlib import Path

KAGGLE_USERNAME = "aftaabkazi"
KAGGLE_KEY = "YOUR_KAGGLE_KEY_HERE"

home = Path.home() / ".kaggle"
home.mkdir(exist_ok=True)
(home / "kaggle.json").write_text(
    json.dumps({"username": KAGGLE_USERNAME, "key": KAGGLE_KEY})
)
os.chmod(home / "kaggle.json", 0o600)
print("Auth OK for", KAGGLE_USERNAME)


In [ ]:
# Cell 3 — download dataset + flatten to Healthy / Rotten
import shutil, subprocess
from pathlib import Path

KAGGLE_SLUG = "muhammad0subhan/fruit-and-vegetable-disease-healthy-vs-rotten"
DATA_ROOT = Path("/content/produce_data")
FLAT = DATA_ROOT / "flat"
DATA_ROOT.mkdir(parents=True, exist_ok=True)

has_imgs = (
    any(DATA_ROOT.rglob("*.jpg"))
    or any(DATA_ROOT.rglob("*.jpeg"))
    or any(DATA_ROOT.rglob("*.png"))
)
if not has_imgs:
    print("Downloading from Kaggle (auto)…")
    rc = subprocess.call(
        [
            "kaggle",
            "datasets",
            "download",
            "-d",
            KAGGLE_SLUG,
            "-p",
            str(DATA_ROOT),
            "--unzip",
        ]
    )
    assert rc == 0, (
        "Download failed. Open the dataset page, click I Agree, re-run Cell 2–3."
    )
else:
    print("Images already present")


def flatten(src: Path, dst: Path):
    healthy_ok = (dst / "Healthy").exists() and any((dst / "Healthy").iterdir())
    rotten_ok = (dst / "Rotten").exists() and any((dst / "Rotten").iterdir())
    if healthy_ok and rotten_ok:
        print("Flat already built")
        return
    for d in ["Healthy", "Rotten"]:
        (dst / d).mkdir(parents=True, exist_ok=True)
    n = {"Healthy": 0, "Rotten": 0}
    for path in src.rglob("*"):
        if path.suffix.lower() not in {".jpg", ".jpeg", ".png"}:
            continue
        blob = f"{path.parent.name.lower()}/{path.name.lower()}"
        if "healthy" in blob and "rotten" not in blob:
            label = "Healthy"
        elif "rotten" in blob:
            label = "Rotten"
        else:
            continue
        out = dst / label / f"{n[label]:06d}{path.suffix.lower()}"
        shutil.copy2(path, out)
        n[label] += 1
    print("Flattened:", n)


src = DATA_ROOT
for p in DATA_ROOT.iterdir():
    if p.is_dir() and p.name not in ("flat", "raw"):
        src = p
        break
flatten(src, FLAT)
print("Ready:", FLAT)


In [ ]:
# Cell 4 — Colab's built-in TensorFlow (do not pip install)
import json, random
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    accuracy_score,
    f1_score,
)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import MobileNetV2, EfficientNetB0, ResNet50

print("TF", tf.__version__)
gpus = tf.config.list_physical_devices("GPU")
print("GPU", gpus)
assert gpus, "Enable GPU: Runtime → Change runtime type → T4 GPU, then Restart"

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

IMG, BATCH = 224, 32
EPOCHS_HEAD, EPOCHS_FINE = 3, 2  # enough for demo; still real training
FLAT = Path("/content/produce_data/flat")
OUT = Path("/content/agrosight_exports")
OUT.mkdir(exist_ok=True)


In [ ]:
# Cell 5 — tf.data pipelines
train_ds = tf.keras.utils.image_dataset_from_directory(
    FLAT,
    validation_split=0.2,
    subset="training",
    seed=SEED,
    image_size=(IMG, IMG),
    batch_size=BATCH,
    label_mode="binary",
)
val_ds = tf.keras.utils.image_dataset_from_directory(
    FLAT,
    validation_split=0.2,
    subset="validation",
    seed=SEED,
    image_size=(IMG, IMG),
    batch_size=BATCH,
    label_mode="binary",
)
print("classes", train_ds.class_names)

AUTOTUNE = tf.data.AUTOTUNE
aug = keras.Sequential(
    [
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.1),
        layers.RandomZoom(0.1),
    ]
)


def prep(ds, training=False):
    ds = ds.map(
        lambda x, y: (tf.cast(x, tf.float32) / 255.0, y),
        num_parallel_calls=AUTOTUNE,
    )
    if training:
        ds = ds.map(
            lambda x, y: (aug(x, training=True), y),
            num_parallel_calls=AUTOTUNE,
        )
    return ds.prefetch(AUTOTUNE)


train, val = prep(train_ds, True), prep(val_ds, False)


In [ ]:
# Cell 6 — train & compare 3 backbones
def build_model(name):
    inp = keras.Input(shape=(IMG, IMG, 3))
    ctor = {
        "MobileNetV2": MobileNetV2,
        "EfficientNetB0": EfficientNetB0,
        "ResNet50": ResNet50,
    }[name]
    base = ctor(include_top=False, weights="imagenet", input_tensor=inp)
    base.trainable = False
    x = layers.GlobalAveragePooling2D()(base.output)
    x = layers.Dropout(0.3)(x)
    out = layers.Dense(1, activation="sigmoid")(x)
    return keras.Model(inp, out, name=name), base


results, best, best_score = [], None, -1.0
for name in ["MobileNetV2", "EfficientNetB0", "ResNet50"]:
    print("\n===", name, "===")
    model, base = build_model(name)
    model.compile(
        optimizer=keras.optimizers.Adam(1e-3),
        loss="binary_crossentropy",
        metrics=["accuracy"],
    )
    model.fit(train, validation_data=val, epochs=EPOCHS_HEAD, verbose=1)
    base.trainable = True
    model.compile(
        optimizer=keras.optimizers.Adam(1e-5),
        loss="binary_crossentropy",
        metrics=["accuracy"],
    )
    model.fit(train, validation_data=val, epochs=EPOCHS_FINE, verbose=1)

    y_true, y_pred = [], []
    for bx, by in val:
        p = model.predict(bx, verbose=0).ravel()
        y_true.extend(by.numpy().ravel().tolist())
        y_pred.extend((p >= 0.5).astype(int).tolist())
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    composite = 0.6 * acc + 0.4 * f1
    row = {
        "name": name,
        "acc": float(acc),
        "f1": float(f1),
        "composite": float(composite),
    }
    results.append(row)
    print(row)
    if composite > best_score:
        best_score, best = composite, model

print(json.dumps(results, indent=2))


In [ ]:
# Cell 7 — charts for Insights page
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar([r["name"] for r in results], [r["composite"] for r in results], color="#0ea5e9")
ax.set_ylabel("Composite (0.6·acc + 0.4·F1)")
ax.set_title("Produce Quality — model comparison")
plt.xticks(rotation=15, ha="right")
plt.tight_layout()
fig.savefig(OUT / "11_produce_model_comparison.png", dpi=150)
plt.show()

y_true, y_pred = [], []
for bx, by in val:
    p = best.predict(bx, verbose=0).ravel()
    y_true.extend(by.numpy().ravel().tolist())
    y_pred.extend((p >= 0.5).astype(int).tolist())

cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=train_ds.class_names,
    yticklabels=train_ds.class_names,
    ax=ax,
)
ax.set_title("Produce Quality confusion matrix")
plt.tight_layout()
fig.savefig(OUT / "12_produce_confusion_matrix.png", dpi=150)
plt.show()
print(classification_report(y_true, y_pred, target_names=train_ds.class_names))


In [ ]:
# Cell 8 — export BINARY TFLite (P(Rotten) sigmoid) — DO NOT use subclass ThreeWay
# Training used x/255.0. App feeds the same. Classes: Healthy=0, Rotten=1 (folder order).
best.save(OUT / "produce_best.keras")
converter = tf.lite.TFLiteConverter.from_keras_model(best)
converter.optimizations = []  # keep fp32
tflite_model = converter.convert()
(OUT / "agrosight_produce.tflite").write_bytes(tflite_model)

# Sanity: fresh-ish vs dark should NOT be identical
interp = tf.lite.Interpreter(model_content=tflite_model)
interp.allocate_tensors()
inp = interp.get_input_details()[0]
out = interp.get_output_details()[0]
for name, arr in [("zeros", np.zeros((1, IMG, IMG, 3), np.float32)), ("ones", np.ones((1, IMG, IMG, 3), np.float32))]:
    interp.set_tensor(inp["index"], arr)
    interp.invoke()
    print(name, "P(Rotten)=", interp.get_tensor(out["index"]).ravel())

acc = accuracy_score(y_true, y_pred)
metrics = {
    "produce_accuracy": round(float(acc) * 100, 2),
    "produce_f1": round(float(f1_score(y_true, y_pred)) * 100, 2),
    "winner": max(results, key=lambda r: r["composite"])["name"],
    "results": results,
    "tf_version": tf.__version__,
    "export": "binary_sigmoid_P_rotten",
    "preprocess": "float32_rgb_div_255",
    "class_order": list(train_ds.class_names),
}
(OUT / "produce_metrics.json").write_text(json.dumps(metrics, indent=2))
print("Wrote", OUT / "agrosight_produce.tflite", "bytes", len(tflite_model))
print(metrics)


In [ ]:
# Cell 9 — download to your PC
from google.colab import files

for f in [
    "agrosight_produce.tflite",
    "11_produce_model_comparison.png",
    "12_produce_confusion_matrix.png",
    "produce_metrics.json",
]:
    files.download(str(OUT / f))

print('''Drop into repo:
  agrosight/public/models/agrosight_produce.tflite
  agrosight/public/assets/11_produce_model_comparison.png
  agrosight/public/assets/12_produce_confusion_matrix.png
Then tell Cursor: produce tflite dropped''')
